# Fine-tuned Chatbot Tim Legal berbasis RAG
# Membangun Sistem RAG

---
## 1. Setup & Instalasi Dependensi

In [21]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [22]:
%%capture

!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps unsloth
!pip install langchain langchain-community langchain-huggingface chromadb pypdf sentence-transformers rank_bm25 duckduckgo-search

In [23]:
import os
import re
import glob
import torch
from kaggle_secrets import UserSecretsClient
from IPython.display import display, Markdown

# Setup Hugging Face Token dari Kaggle Secrets
try:
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN berhasil dimuat dari Kaggle Secrets")
except Exception as e:
    print(f"Gagal memuat HF_TOKEN dari Kaggle Secrets: {e}")

HF_TOKEN berhasil dimuat dari Kaggle Secrets


---
## 2. Memuat Model Hasil Pelatihan Kriteria 1 (Model GRPO)

In [24]:
HF_USERNAME = "afihsan"
REPO_NAME_GRPO = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-grpo"
REPO_NAME_SFT = f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-finetuned"

from unsloth import FastLanguageModel

print("Memuat Model untuk RAG Pipeline...")
try:
    print(f"Mencoba memuat model GRPO: {REPO_NAME_GRPO}")
    model_rag, tokenizer_rag = FastLanguageModel.from_pretrained(
        model_name=REPO_NAME_GRPO,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    print("Berhasil memuat model GRPO!")
except Exception as e:
    print(f"Model GRPO belum tersedia ({e}), beralih ke Base SFT Model: {REPO_NAME_SFT}")
    model_rag, tokenizer_rag = FastLanguageModel.from_pretrained(
        model_name=REPO_NAME_SFT,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    print("Berhasil memuat model Base SFT!")

FastLanguageModel.for_inference(model_rag)

Memuat Model untuk RAG Pipeline...
Mencoba memuat model GRPO: afihsan/qwen2.5-1.5b-legal-chatbot-grpo
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe03ba9780>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe032981a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe03ba9780>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe032981a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe03ba9780>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7fbe032981a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.So

total weights      : 1.045 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.281 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  11.42 GiB  weights  0.480 GiB  free 10.938 GiB  reserve 10.920 GiB
  cuda:1  budget  12.55 GiB  weights  0.566 GiB  free 11.982 GiB  reserve 11.281 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 12.686 -> 11.418 GiB, cuda:1 13.942 -> 12.548 GiB (memory the quantiser keeps for its own load-time buffers)
note: tied embeddings: model.embed_tokens pinned with the head so accelerate does not have to mirror the weight across devices


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Berhasil memuat model GRPO!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

---
## 3. Memuat 4 Dokumen Hukum PDF

Wajib memuat seluruh 4 file PDF resmi yang disediakan:
1. `PP Nomor 35 Tahun 2021.pdf` (PKWT, Alih Daya, Waktu Kerja, Hubungan Kerja, dan PHK)
2. `PP Nomor 5 Tahun 2021.pdf` (Penyelenggaraan Perizinan Berusaha Berbasis Risiko)
3. `PP Nomor 51 Tahun 2023.pdf` (Pengupahan)
4. `UU Nomor 6 Tahun 2023.pdf` (Penetapan Cipta Kerja menjadi UU)

In [25]:
from langchain_community.document_loaders import PyPDFLoader

# Cari file PDF di direktori kerja atau folder Document RAG
pdf_paths = glob.glob("./Document RAG/*.pdf") + glob.glob("./*.pdf") + glob.glob("/kaggle/input/**/*.pdf", recursive=True)
pdf_paths = list(set([p for p in pdf_paths if os.path.basename(p).endswith(".pdf") and not p.endswith("Submission Chatbot.pdf")]))

print(f"Ditemukan {len(pdf_paths)} dokumen PDF hukum:")
for p in sorted(pdf_paths):
    print(f"   - {os.path.basename(p)}")

raw_docs = []
for path in pdf_paths:
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()
        raw_docs.extend(docs)
        print(f"{os.path.basename(path)}: {len(docs)} halaman dimuat")
    except Exception as e:
        print(f"Error loading {path}: {e}")

print(f"\nTotal keseluruhan halaman dokumen: {len(raw_docs)}")

Ditemukan 4 dokumen PDF hukum:
   - PP Nomor 35 Tahun 2021.pdf
   - PP Nomor 5 Tahun 2021.pdf
   - PP Nomor 51 Tahun 2023.pdf
   - UU Nomor 6 Tahun 2023.pdf
PP Nomor 51 Tahun 2023.pdf: 27 halaman dimuat
PP Nomor 35 Tahun 2021.pdf: 56 halaman dimuat
PP Nomor 5 Tahun 2021.pdf: 739 halaman dimuat
UU Nomor 6 Tahun 2023.pdf: 1127 halaman dimuat

Total keseluruhan halaman dokumen: 1949


---
## 4. Metadata Enrichment

Melakukan pengayaan metadata pada setiap dokumen:
- `source`: Nama file dokumen
- `doc_type`: Jenis regulasi (Peraturan Pemerintah / Undang-Undang)
- `nomor_tahun`: Nomor dan tahun regulasi (misal PP 35/2021, UU 6/2023)
- `topik`: Topik hukum utama (Ketenagakerjaan & PHK, Pengupahan, Perizinan Berusaha, Cipta Kerja)
- `page`: Nomor halaman asli PDF

In [26]:
def enrich_document_metadata(docs):
    enriched = []
    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", ""))
        
        if "PP Nomor 35 Tahun 2021" in filename:
            doc_type = "Peraturan Pemerintah"
            nomor_tahun = "PP No. 35 Tahun 2021"
            topik = "Ketenagakerjaan, Waktu Kerja, Lembur, dan PHK"
        elif "PP Nomor 5 Tahun 2021" in filename:
            doc_type = "Peraturan Pemerintah"
            nomor_tahun = "PP No. 5 Tahun 2021"
            topik = "Penyelenggaraan Perizinan Berusaha Berbasis Risiko"
        elif "PP Nomor 51 Tahun 2023" in filename:
            doc_type = "Peraturan Pemerintah"
            nomor_tahun = "PP No. 51 Tahun 2023"
            topik = "Pengupahan dan Upah Minimum"
        elif "UU Nomor 6 Tahun 2023" in filename:
            doc_type = "Undang-Undang"
            nomor_tahun = "UU No. 6 Tahun 2023"
            topik = "Penetapan Perppu Cipta Kerja Menjadi Undang-Undang"
        else:
            doc_type = "Regulasi Hukum"
            nomor_tahun = filename
            topik = "Hukum Umum"
            
        doc.metadata["doc_type"] = doc_type
        doc.metadata["nomor_tahun"] = nomor_tahun
        doc.metadata["topik"] = topik
        doc.metadata["filename"] = filename
        enriched.append(doc)
    return enriched

enriched_docs = enrich_document_metadata(raw_docs)
print("Metadata Enrichment berhasil diterapkan ke seluruh halaman!")
print("Contoh metadata dokumen:", enriched_docs[0].metadata)

Metadata Enrichment berhasil diterapkan ke seluruh halaman!
Contoh metadata dokumen: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2023-11-10T12:26:08+07:00', 'moddate': '2023-11-10T13:00:46+07:00', 'source': '/kaggle/input/datasets/afihsan/dokumen-rag/Document RAG/PP Nomor 51 Tahun 2023.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'doc_type': 'Peraturan Pemerintah', 'nomor_tahun': 'PP No. 51 Tahun 2023', 'topik': 'Pengupahan dan Upah Minimum', 'filename': 'PP Nomor 51 Tahun 2023.pdf'}


---
## 5. Parent-Child Chunking & Retriever

In [27]:
# Import text splitter yang kompatibel dengan LangChain versi baru maupun lama
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document

import uuid

# 1. Definisi Splitter untuk Parent Chunks dan Child Chunks
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=150,
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

# 2. Proses pembuatan Parent dan Child Chunks dengan relasi ID
parent_docs = parent_splitter.split_documents(enriched_docs)
parent_store = {}  
child_docs = []

for parent_doc in parent_docs:
    parent_id = str(uuid.uuid4())
    parent_doc.metadata["parent_id"] = parent_id
    parent_store[parent_id] = parent_doc
    
    sub_chunks = child_splitter.split_text(parent_doc.page_content)
    for sub in sub_chunks:
        child_doc = Document(
            page_content=sub,
            metadata={**parent_doc.metadata, "parent_id": parent_id}
        )
        child_docs.append(child_doc)

print(f"Parent Chunks dibuat : {len(parent_docs)}")
print(f"Child Chunks dibuat  : {len(child_docs)}")

# 3. Inisialisasi Open-Source Embedding & Vector Database (ChromaDB)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
)

print("Membuat ChromaDB Vector Store untuk Child Chunks...")
vector_store = Chroma.from_documents(
    documents=child_docs,
    embedding=embedding_model,
    collection_name="legal_child_chunks",
    persist_directory="./chroma_db",
)
print("Vector Database ChromaDB berhasil dibangun!")

Parent Chunks dibuat : 1992
Child Chunks dibuat  : 6575


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Membuat ChromaDB Vector Store untuk Child Chunks...
Vector Database ChromaDB berhasil dibangun!


---
## 6. Ensemble Retriever: BM25 (Keyword) + ChromaDB (Semantic)

In [29]:
from langchain_community.retrievers import BM25Retriever

# Import EnsembleRetriever dengan fallback ke langchain-classic atau implementasi RRF mandiri
try:
    from langchain.retrievers import EnsembleRetriever
except (ImportError, ModuleNotFoundError):
    try:
        from langchain_classic.retrievers import EnsembleRetriever
    except (ImportError, ModuleNotFoundError):
        class EnsembleRetriever:
            """
            Implementasi mandiri Ensemble Retriever berbasis Reciprocal Rank Fusion (RRF)
            kompatibel dengan LangChain tanpa dependensi modul eksternal yang rentan versi.
            """
            def __init__(self, retrievers, weights=None, c=60):
                self.retrievers = retrievers
                self.weights = weights or [1.0 / len(retrievers)] * len(retrievers)
                self.c = c

            def invoke(self, query):
                return self.get_relevant_documents(query)

            def get_relevant_documents(self, query):
                rrf_scores = {}
                doc_map = {}
                for retriever, weight in zip(self.retrievers, self.weights):
                    if hasattr(retriever, 'invoke'):
                        docs = retriever.invoke(query)
                    elif hasattr(retriever, 'get_relevant_documents'):
                        docs = retriever.get_relevant_documents(query)
                    else:
                        docs = []
                    for rank, doc in enumerate(docs):
                        doc_id = getattr(doc, 'page_content', str(doc))
                        if doc_id not in rrf_scores:
                            rrf_scores[doc_id] = 0.0
                            doc_map[doc_id] = doc
                        rrf_scores[doc_id] += weight / (self.c + rank + 1)
                sorted_docs = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)
                return [doc_map[k] for k in sorted_docs]

# 1. BM25 Retriever untuk pencarian kata kunci eksak pada parent chunks
bm25_retriever = BM25Retriever.from_documents(parent_docs)
bm25_retriever.k = 5

# 2. Vector Store Retriever untuk pencarian semantik
semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# 3. Ensemble Retriever menggabungkan Keyword + Semantic dengan bobot (weights=[0.5, 0.5])
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.5, 0.5],
)

# Fungsi untuk menyelesaikan Parent Document jika hasil dari child chunk
def resolve_parent_documents(retrieved_docs):
    resolved = []
    seen_ids = set()
    for doc in retrieved_docs:
        p_id = doc.metadata.get("parent_id")
        if p_id and p_id in parent_store:
            if p_id not in seen_ids:
                resolved.append(parent_store[p_id])
                seen_ids.add(p_id)
        else:
            resolved.append(doc)
    return resolved

print("Ensemble Retriever (BM25 + Semantic ChromaDB) siap digunakan (k >= 5)!")

Ensemble Retriever (BM25 + Semantic ChromaDB) siap digunakan (k >= 5)!


---
## 7. HyDE: Hypothetical Document Embeddings

In [30]:
def generate_hyde_documents(query, model, tokenizer, num_hypotheses=2):
    hypotheses = []
    for i in range(num_hypotheses):
        prompt = f"<|im_start|>user\nTulislah jawaban hipotetis singkat mengenai pertanyaan hukum berikut berdasarkan regulasi ketenagakerjaan Indonesia:\nPertanyaan: {query}\nJawaban:<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.8, top_p=0.9)
        hypo_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        hypotheses.append(hypo_text)
    return hypotheses

print("Fungsi HyDE (minimal 2 jawaban halusinasi) berhasil didefinisikan!")

Fungsi HyDE (minimal 2 jawaban halusinasi) berhasil didefinisikan!


---
## 8. Cross-Encoder Reranker & Top-K Selection

In [31]:
from sentence_transformers import CrossEncoder

# Inisialisasi model Cross-Encoder Reranker
print("Memuat Model Cross-Encoder Reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-Encoder Reranker berhasil dimuat!")

def rerank_documents(query, documents, top_k=3):
    if not documents:
        return [], []
        
    pairs = [[query, doc.page_content] for doc in documents]
    scores = reranker.predict(pairs)
    
    scored_docs = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    top_scored = scored_docs[:top_k]
    top_docs = [doc for score, doc in top_scored]
    top_scores = [score for score, doc in top_scored]
    
    return top_docs, top_scores

Memuat Model Cross-Encoder Reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder Reranker berhasil dimuat!


---
## 9. Dynamic Relevance Score Thresholding & DuckDuckGo Fallback

In [34]:
from duckduckgo_search import DDGS

# Threshold relevance score model Cross-Encoder ms-marco (logits)
RELEVANCE_THRESHOLD = -7.5

def fallback_duckduckgo_search(query, max_results=3):
    print("Menjalankan DuckDuckGo Search untuk mencari jawaban dari internet...")
    search_results = []
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query + " hukum indonesia regulasi", max_results=max_results)
            for res in results:
                doc = Document(
                    page_content=res.get("body", ""),
                    metadata={
                        "source": res.get("href", "Internet"),
                        "nomor_tahun": "DuckDuckGo Web Search",
                        "topik": res.get("title", "Web Search")
                    }
                )
                search_results.append(doc)
    except Exception as e:
        print(f"DuckDuckGo search error: {e}")
    return search_results

print(f"Logika Thresholding dan DuckDuckGo Fallback terkonfigurasi (Threshold: {RELEVANCE_THRESHOLD})!")

Logika Thresholding dan DuckDuckGo Fallback terkonfigurasi (Threshold: -7.5)!


---
## 10. Pipeline RAG Lengkap dengan Format Reasoning `<think>` & Sitasi

In [35]:
def advanced_rag_pipeline(query, top_k=3):
    print(f"\nMemproses Pertanyaan: '{query}'")
    
    # 1. HyDE Transformation (Minimal 2 Hipotesis)
    print("1. Menghasilkan 2 jawaban halusinasi awal via HyDE...")
    hypotheses = generate_hyde_documents(query, model_rag, tokenizer_rag, num_hypotheses=2)
    combined_query = f"{query} {' '.join(hypotheses)}"
    
    # 2. Ensemble Retrieval (Keyword + Semantic)
    print("2. Mengambil dokumen via Ensemble Retriever (BM25 + ChromaDB)... ")
    retrieved_raw = ensemble_retriever.invoke(combined_query)
    retrieved_docs = resolve_parent_documents(retrieved_raw)
    print(f"Ditemukan {len(retrieved_docs)} dokumen kandidat.")
    
    # 3. Cross-Encoder Reranking
    print("3. Melakukan Reranking dengan Cross-Encoder...")
    top_docs, top_scores = rerank_documents(query, retrieved_docs, top_k=top_k)
    
    top1_score = top_scores[0] if top_scores else -99.0
    print(f"Skor Relevansi Top-1: {top1_score:.3f}")
    
    # 4. Conditional Check (Thresholding & DuckDuckGo Fallback)
    if top1_score >= RELEVANCE_THRESHOLD:
        print("Skor Top-1 memenuhi threshold. Menggunakan basis data dokumen lokal (UU/PP).")
        final_context_docs = top_docs
        source_mode = "Dokumen Resmi Internal (UU / PP)"
    else:
        print("Skor Top-1 di bawah threshold! Mengabaikan dokumen lokal dan beralih ke DuckDuckGo...")
        web_docs = fallback_duckduckgo_search(query)
        final_context_docs = web_docs if web_docs else top_docs
        source_mode = "DuckDuckGo Web Search"
        
    # 5. Susun Konteks Dokumen & Sitasi
    context_texts = []
    citations = []
    for i, doc in enumerate(final_context_docs, 1):
        nomor = doc.metadata.get("nomor_tahun", "Dokumen")
        page = doc.metadata.get("page", "-")
        context_texts.append(f"[Dokumen {i} - {nomor} (Hal. {page})]:\n{doc.page_content}")
        citations.append(f"{i}. {nomor} (Halaman: {page}) - Topik: {doc.metadata.get('topik', '-')}")
        
    full_context = "\n\n".join(context_texts)
    
    # 6. Prompt Engineering untuk Model GRPO
    prompt = f"""<|im_start|>system\n"""
    prompt += """Anda adalah asisten AI Tim Legal perusahaan. Tugas Anda menjawab pertanyaan berdasarkan konteks dokumen hukum yang diberikan.\n"""
    prompt += """Wajib ikuti aturan ini:\n1. Uraikan analisis dan proses penalaran hukum Anda di dalam tag <think> ... </think>.\n2. Setelah tag </think>, berikan jawaban akhir yang tegas, jelas, dan sebutkan dasar hukumnya.<|im_end|>\n"""
    prompt += f"""<|im_start|>user\nKonteks Dokumen:\n{full_context}\n\nPertanyaan:\n{query}<|im_end|>\n<|im_start|>assistant\n<think>\n"""
    
    # 7. Model Generation
    inputs = tokenizer_rag(prompt, return_tensors="pt").to("cuda")
    outputs = model_rag.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.7,
        top_p=0.9,
    )
    
    generated_raw = tokenizer_rag.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    full_answer = "<think>\n" + generated_raw.lstrip()
    
    return full_answer, citations, source_mode, top1_score

---
## 11. Pengujian Test Case Wajib

In [36]:
test_query = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

answer, citations, mode, score = advanced_rag_pipeline(test_query)

display(Markdown(f"### HASIL EVALUASI TEST CASE WAJIB"))
display(Markdown(f"**Pertanyaan**: *{test_query}*"))
display(Markdown(f"**Mode Sumber**: `{mode}` (Skor Relevansi: `{score:.3f}`)"))
display(Markdown(f"---\n### Jawaban Model (Reasoning & Final Answer):\n\n{answer}"))
display(Markdown(f"---\n### Sitasi Dokumen Hukum Terkait:\n" + "\n".join([f"- {c}" for c in citations])))

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Memproses Pertanyaan: 'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?'
1. Menghasilkan 2 jawaban halusinasi awal via HyDE...


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Mengambil dokumen via Ensemble Retriever (BM25 + ChromaDB)... 
Ditemukan 7 dokumen kandidat.
3. Melakukan Reranking dengan Cross-Encoder...
Skor Relevansi Top-1: -3.156
Skor Top-1 memenuhi threshold. Menggunakan basis data dokumen lokal (UU/PP).


### HASIL EVALUASI TEST CASE WAJIB

**Pertanyaan**: *Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?*

**Mode Sumber**: `Dokumen Resmi Internal (UU / PP)` (Skor Relevansi: `-3.156`)

---
### Jawaban Model (Reasoning & Final Answer):

<think>
Dokumen 1: UU No. 6 Tahun 2023 (Hal. 558) menyatakan bahwa pengusaha wajib memberi waktu istirahat dan cuti kepada pekerja/buruh, dan waktu istirahat sebagaimana dimaksud pada ayat (1) huruf a wajib diberikan kepada pekerja/buruh paling sedikit meliputi: istirahat antara jam kerja, paling sedikit setengah jam setelah bekerja selama 4 (empat) jam terus-menerus, dan waktu istirahat tersebut tidak termasuk jam kerja. Cuti sebagaimana dimaksud pada ayat (1) huruf b yang wajib diberikan kepada pekerja/buruh, yaitu cuti tahunan, paling sedikit 12 (dua belas) hari kerja setelah pekerja/buruh yang bersangkutan bekerja selama 12 (dua belas) bulan secara terus menerus. Pelaksanaan cuti tahunan sebagaimana dimaksud pada ayat (3) diatur dalam Perjanjian Kerja, Peraturan Perusahaan, atau Perjanjian Kerja Bersama.
Dokumen 2: UU No. 6 Tahun 2023 (Hal. 558) menyatakan bahwa pengusaha wajib memberi waktu istirahat dan cuti kepada pekerja/buruh, dan waktu istirahat sebagaimana dimaksud pada ayat (1) huruf a wajib diberikan kepada pekerja/buruh paling sedikit meliputi: istirahat antara jam kerja, paling sedikit setengah jam setelah bekerja selama 4 (empat) jam terus-menerus, dan waktu istirahat tersebut tidak termasuk jam kerja. Cuti sebagaimana dimaksud pada ayat (1) huruf b yang wajib diberikan kepada pekerja/buruh, yaitu cuti tahunan, paling sedikit 12 (dua belas) hari kerja setelah pekerja/buruh yang bersangkutan bekerja selama 12 (dua belas) bulan secara terus menerus. Pelaksanaan cuti tahunan sebagaimana dimaksud pada ayat (3) diatur dalam Perjanjian Kerja, Peraturan Perusahaan, atau Perjanjian Kerja Bersama.
Dokumen 3: PP No. 35 Tahun 2021 (Hal. 10) menyatakan bahwa PKWT harus dicatatkan

---
### Sitasi Dokumen Hukum Terkait:
- 1. UU No. 6 Tahun 2023 (Halaman: 558) - Topik: Penetapan Perppu Cipta Kerja Menjadi Undang-Undang
- 2. UU No. 6 Tahun 2023 (Halaman: 558) - Topik: Penetapan Perppu Cipta Kerja Menjadi Undang-Undang
- 3. PP No. 35 Tahun 2021 (Halaman: 10) - Topik: Ketenagakerjaan, Waktu Kerja, Lembur, dan PHK

---
## 12. Interface Interaktif: Interactive Python Loop

In [37]:
def interactive_legal_chatbot():
    display(Markdown("# Chatbot Tim Legal Internal Perusahaan (Advanced RAG)"))
    display(Markdown("Silakan ajukan pertanyaan hukum ketenagakerjaan atau regulasi perizinan. Ketik `'exit'` atau `'quit'` untuk keluar.\n---"))
    
    while True:
        try:
            user_input = input("\n Pertanyaan Anda: ")
        except (EOFError, KeyboardInterrupt):
            print("\nSesi diakhiri.")
            break
            
        if user_input.strip().lower() in ["exit", "quit", "keluar"]:
            display(Markdown("**Terima kasih telah menggunakan asisten hukum AI. Sesi selesai.**"))
            break
            
        if not user_input.strip():
            continue
            
        display(Markdown(f"\n**Pertanyaan:** {user_input}"))
        
        try:
            resp, cites, src_mode, top_score = advanced_rag_pipeline(user_input)
            display(Markdown(f"\n**Sumber Informasi:** `{src_mode}` (Score: `{top_score:.3f}`)"))
            display(Markdown(f"\n### Jawaban:\n\n{resp}"))
            display(Markdown(f"\n**Sitasi Dokumen:**\n" + "\n".join([f"- {c}" for c in cites])))
            display(Markdown("---"))
        except Exception as err:
            display(Markdown(f"> Terjadi kesalahan: {err}"))

# Untuk menjalankan chatbot interaktif, panggil:
# interactive_legal_chatbot()
print("Interface interaktif 'interactive_legal_chatbot()' siap dijalankan!")

Interface interaktif 'interactive_legal_chatbot()' siap dijalankan!


In [38]:
interactive_legal_chatbot()

# Chatbot Tim Legal Internal Perusahaan (Advanced RAG)

Silakan ajukan pertanyaan hukum ketenagakerjaan atau regulasi perizinan. Ketik `'exit'` atau `'quit'` untuk keluar.
---


 Pertanyaan Anda:  Berapa batas maksimal waktu kerja lembur dalam sehari menurut PP 35/2021?



**Pertanyaan:** Berapa batas maksimal waktu kerja lembur dalam sehari menurut PP 35/2021?

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


### Jawaban:

<think>
Dokumen 1 menyatakan bahwa pengusaha yang mempekerjakan Pekerja/Buruh melebihi waktu kerja sebagaimana dimaksud dalam Pasal 29 ayat (2) wajib membayar Upah Kerja Lembur. Pasal 30 menyatakan bahwa Upah Kerja Lembur tidak dapat digantikan dalam bentuk uang. Dokumen 2 menyatakan bahwa pengusaha wajib memberi waktu istirahat dan cuti, dan Pasal 79 menyatakan bahwa pengusaha wajib memberi waktu istirahat dan cuti. Pasal 79 ayat (1) menyatakan bahwa pengusaha wajib memberi waktu istirahat, dan Pasal 79 ayat (2) menyatakan bahwa pengusaha wajib memberi cuti. Pasal 79 ayat (3) menyatakan bahwa cuti wajib diberikan kepada Pekerja/Buruh, dan Pasal 79 ayat (4) menyatakan bahwa cuti wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (5) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (6) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (7) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (8) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (9) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (10) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (11) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (12) menyatakan bahwa cuti tahunan wajib diberikan kepada Pekerja/Buruh, yaitu cuti tahunan. Pasal 79 ayat (13) menyatakan bahwa cuti tahun


**Sitasi Dokumen:**
- 1. PP No. 35 Tahun 2021 (Halaman: 17) - Topik: Ketenagakerjaan, Waktu Kerja, Lembur, dan PHK
- 2. UU No. 6 Tahun 2023 (Halaman: 558) - Topik: Penetapan Perppu Cipta Kerja Menjadi Undang-Undang
- 3. UU No. 6 Tahun 2023 (Halaman: 557) - Topik: Penetapan Perppu Cipta Kerja Menjadi Undang-Undang

---


 Pertanyaan Anda:  exit


**Terima kasih telah menggunakan asisten hukum AI. Sesi selesai.**